In [2]:
import warnings
warnings.filterwarnings("ignore", message="The default value of `allowed_objects`")

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [10]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    temperature=0.8,
)

agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions. Provide links to found flights."
)

In [13]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "2"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on June 25th, 2026")]},
    config
    )

In [14]:
from pprint import pprint

pprint(response)
print(response["messages"][-1].content)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on June 25th, 2026', additional_kwargs={}, response_metadata={}, id='df7b5c97-7959-404e-aeaf-a5889cc873da'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants a direct flight from San Francisco to Tokyo on June 25th, 2026. Let me check the tools available. There\'s a flight search function called search-flight. The parameters needed are flyFrom, flyTo, departureDate, and possibly others.\n\nFirst, I need to confirm the locations. San Francisco\'s airport code is SFO, and Tokyo has multiple airports, but the main ones are HND (Haneda) and NRT (Narita). Since the user didn\'t specify, maybe I should consider both, but the tool might handle city names. However, the function\'s description says it resolves airport codes, so using city names should work. But to be precise, maybe inputting the city names is better here.\n\nThe departure date is June 25th, 202